# GPU 02 - Segmentation Model Training at 496x768

Trains the five segmentation models on the same 59/15 split at native 496x768, using the residuals cached in gpu_01.

| Model | Input | Params | Loss | Learning rate |
|-------|-------|--------|------|---------------|
| Plain U-Net | 1ch | 7.77M | Dice+BCE | 0.001 |
| Hybrid U-Net | 2ch | 7.77M | Dice+BCE | 0.001 |
| OCT-F Backbone | 1ch | 1.98M | Dice+BCE | encoder 1e-5, decoder 1e-3 |
| Modified OCT-F 2ch | 2ch | 1.98M | Dice+BCE | encoder 1e-5, decoder 1e-3 |
| Soft Erosion Hybrid | 2ch | 7.77M | Dice+BCE+Erosion | 0.001 |

## 1. Setup

Paths are relative to the project root. Seed is 40494109 for every split and weight initialisation.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pickle, time

# project root (notebook runs from gpu_notebooks/)
PROJECT_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))

# raw data
ENDPOINT_FOLDER = os.path.join(PROJECT_ROOT, 'data',
                    'endpoint')
ANNO_FOLDER = os.path.join(PROJECT_ROOT, 'data', 'annotated',
               'endpoint_annotated')

# generated by gpu_01
MASK_FOLDER = os.path.join(PROJECT_ROOT, 'data', 'masks')

# reconstruction
OCTF_CODE = os.path.join(PROJECT_ROOT, 'reconstruction',
             'OCT-F_Reconstruction', 'code')
FINETUNE_CKPT = os.path.join(PROJECT_ROOT, 'reconstruction',
                 'fine_tuned_reconstruction.pth')
RESIDUAL_CACHE = os.path.join(PROJECT_ROOT, 'reconstruction',
                  'residual_cache.pkl')
CKPT_DIR = os.path.join(PROJECT_ROOT, 'reconstruction', 'checkpoints')

# shared code first so code/datasets.py and code/losses.py are not
# shadowed by the OCT-F files of the same name
sys.path.insert(0, OCTF_CODE)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

# outputs
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
CURVES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'training_curves')
IMAGES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'images')

for d in [MASK_FOLDER, CKPT_DIR, FIGURES_DIR, CURVES_DIR, IMAGES_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 40494109
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

# native OCT geometry, avoids the aspect distortion of a square resize
IMG_H, IMG_W = 496, 768
IMG_SIZE = (IMG_H, IMG_W)

# kernels scale with the vertical dimension
PP_KERNEL = int(15 * IMG_H / 256)
PP_KERNEL = PP_KERNEL if PP_KERNEL % 2 == 1 else PP_KERNEL - 1
EROSION_KERNEL = int(5 * IMG_H / 256)
EROSION_KERNEL = EROSION_KERNEL if EROSION_KERNEL % 2 == 1 else EROSION_KERNEL - 1
print(f"Image size: {IMG_H}x{IMG_W} | PP kernel: {PP_KERNEL} | Erosion kernel: {EROSION_KERNEL}")

# gpu_01 must have run first
for path, name in [(ENDPOINT_FOLDER, 'Endpoint images'),
                   (ANNO_FOLDER, 'Annotated images'),
                   (MASK_FOLDER, 'Lesion masks (gpu_01)'),
                   (FINETUNE_CKPT, 'Fine-tuned OCT-F (gpu_01)'),
                   (RESIDUAL_CACHE, 'Residual cache (gpu_01)')]:
    assert os.path.exists(path), f"MISSING: {name} at {path}"
    print(f"  Found: {name}")
print("All gpu_01 outputs verified.")

## 2. Shared Code and Residual Cache

All five models use the same modules from `code/` and the same cached residuals.

In [ ]:
from architectures import (SimpleUNet, HybridUNet, OCTFSegmentation,
                           ModifiedOCTFSegmentation)
from datasets import SingleChannelDataset, HybridDataset
from losses import DiceBCELoss, SoftErosionLoss
from train import train_model

with open(RESIDUAL_CACHE, 'rb') as f:
    residual_cache = pickle.load(f)

n_masks = len([f for f in os.listdir(MASK_FOLDER) if f.endswith('.png')])
shapes = {r.shape for r in residual_cache.values()}
print(f"Residuals loaded: {len(residual_cache)}  shapes: {shapes}")
print(f"Masks available: {n_masks}")
assert len(residual_cache) == 74, "expected 74 cached residuals"
assert n_masks == 74, "expected 74 masks"

## 3. Train and Validation Split

74 annotated images split 59 train / 15 validation, seed 40494109. Every model uses this split.

In [ ]:
valid_pairs = sorted([f for f in os.listdir(ANNO_FOLDER) if f.endswith('.tif')])

np.random.seed(SEED)
shuffled = valid_pairs.copy()
np.random.shuffle(shuffled)

split_idx = int(0.8 * len(shuffled))
train_files = shuffled[:split_idx]
val_files = shuffled[split_idx:]

assert len(train_files) == 59 and len(val_files) == 15
assert set(train_files).isdisjoint(val_files)
assert all(f in residual_cache for f in train_files + val_files)

print(f"Train: {len(train_files)}  Validation: {len(val_files)}")
print(f"\nValidation files:")
for f in val_files:
    print(f"  {f}")

## 4. Data Loaders

Single-channel loaders for models 1 and 3, two-channel loaders for models 2, 4 and 5. Batch size 4, horizontal flip on training only. Residuals are already 496x768 so they are used without resampling.

In [ ]:
BATCH_SIZE = 4

single_train = SingleChannelDataset(ENDPOINT_FOLDER, MASK_FOLDER, train_files,
                                    img_size=IMG_SIZE, augment=True)
single_val = SingleChannelDataset(ENDPOINT_FOLDER, MASK_FOLDER, val_files,
                                  img_size=IMG_SIZE, augment=False)
hybrid_train = HybridDataset(ENDPOINT_FOLDER, MASK_FOLDER, train_files,
                             residual_cache, img_size=IMG_SIZE, augment=True)
hybrid_val = HybridDataset(ENDPOINT_FOLDER, MASK_FOLDER, val_files,
                           residual_cache, img_size=IMG_SIZE, augment=False)

single_train_loader = DataLoader(single_train, batch_size=BATCH_SIZE, shuffle=True)
single_val_loader = DataLoader(single_val, batch_size=BATCH_SIZE, shuffle=False)
hybrid_train_loader = DataLoader(hybrid_train, batch_size=BATCH_SIZE, shuffle=True)
hybrid_val_loader = DataLoader(hybrid_val, batch_size=BATCH_SIZE, shuffle=False)

x1, y1 = next(iter(single_train_loader))
x2, y2 = next(iter(hybrid_train_loader))
print(f"Single channel batch: {tuple(x1.shape)} -> {tuple(y1.shape)}")
print(f"Two channel batch:    {tuple(x2.shape)} -> {tuple(y2.shape)}")

MAX_EPOCHS = 150
PATIENCE = 10
print(f"\nMax epochs {MAX_EPOCHS}, patience {PATIENCE}, batch size {BATCH_SIZE}")

## 5. Model 1 - Plain U-Net

1-channel input, trained from scratch. Baseline without the residual.

In [ ]:
torch.manual_seed(SEED)
model1 = SimpleUNet().to(device)

optimizer1 = optim.Adam(model1.parameters(), lr=0.001)
scheduler1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer1, mode='min',
                                                  patience=5, factor=0.5)

history1 = train_model(
    model1, single_train_loader, single_val_loader, DiceBCELoss(),
    optimizer1, scheduler1, MAX_EPOCHS, PATIENCE,
    os.path.join(CKPT_DIR, 'plain_unet.pth'), 'Plain U-Net', device,
    PP_KERNEL, EROSION_KERNEL)

## 6. Model 2 - Hybrid U-Net

2-channel input, image and residual fused at the first convolution.

In [ ]:
torch.manual_seed(SEED)
model2 = HybridUNet().to(device)

optimizer2 = optim.Adam(model2.parameters(), lr=0.001)
scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min',
                                                  patience=5, factor=0.5)

history2 = train_model(
    model2, hybrid_train_loader, hybrid_val_loader, DiceBCELoss(),
    optimizer2, scheduler2, MAX_EPOCHS, PATIENCE,
    os.path.join(CKPT_DIR, 'hybrid_unet.pth'), 'Hybrid U-Net', device,
    PP_KERNEL, EROSION_KERNEL)

## 7. Fine-Tuned OCT-F Encoder

Loaded once. Models 3 and 4 deep-copy the encoder blocks, so neither affects the other.

In [ ]:
from architectureV2Res_new import ArchitectureV2

ft_model = ArchitectureV2().to(device)
ft_ckpt = torch.load(FINETUNE_CKPT, map_location=device, weights_only=False)
ft_model.load_state_dict(ft_ckpt['model_state_dict'])
ft_model.eval()

print(f"Loaded fine-tuned OCT-F from epoch {ft_ckpt['epoch']}")
print(f"Held-out SSIM: {ft_ckpt['held_out_ssim']:.4f}")


def split_parameters(model):
    """Separate pretrained encoder parameters from new decoder parameters."""
    encoder = [p for block in (model.enc1, model.enc2, model.enc3, model.enc4)
               for p in block.parameters()]
    encoder_ids = {id(p) for p in encoder}
    decoder = [p for p in model.parameters() if id(p) not in encoder_ids]
    return encoder, decoder

## 8. Model 3 - OCT-F Backbone

1-channel input, pretrained encoder at 1e-5 and new decoder at 1e-3.

In [ ]:
torch.manual_seed(SEED)
model3 = OCTFSegmentation(ft_model).to(device)
enc3, dec3 = split_parameters(model3)

optimizer3 = optim.Adam([{'params': enc3, 'lr': 1e-5},
                         {'params': dec3, 'lr': 1e-3}])
scheduler3 = optim.lr_scheduler.ReduceLROnPlateau(optimizer3, mode='min',
                                                  patience=5, factor=0.5)

print(f"Encoder {sum(p.numel() for p in enc3):,} params at 1e-5")
print(f"Decoder {sum(p.numel() for p in dec3):,} params at 1e-3")

history3 = train_model(
    model3, single_train_loader, single_val_loader, DiceBCELoss(),
    optimizer3, scheduler3, MAX_EPOCHS, PATIENCE,
    os.path.join(CKPT_DIR, 'octf_backbone.pth'), 'OCT-F Backbone', device,
    PP_KERNEL, EROSION_KERNEL)

## 9. Model 4 - Modified OCT-F 2ch

2-channel input. Only the first convolution and its shortcut are replaced, 704 new parameters.

In [ ]:
torch.manual_seed(SEED)
model4 = ModifiedOCTFSegmentation(ft_model).to(device)
enc4, dec4 = split_parameters(model4)

optimizer4 = optim.Adam([{'params': enc4, 'lr': 1e-5},
                         {'params': dec4, 'lr': 1e-3}])
scheduler4 = optim.lr_scheduler.ReduceLROnPlateau(optimizer4, mode='min',
                                                  patience=5, factor=0.5)

print(f"Encoder {sum(p.numel() for p in enc4):,} params at 1e-5")
print(f"Decoder {sum(p.numel() for p in dec4):,} params at 1e-3")

history4 = train_model(
    model4, hybrid_train_loader, hybrid_val_loader, DiceBCELoss(),
    optimizer4, scheduler4, MAX_EPOCHS, PATIENCE,
    os.path.join(CKPT_DIR, 'modified_octf_2ch.pth'), 'Modified OCT-F 2ch', device,
    PP_KERNEL, EROSION_KERNEL)

## 10. Model 5 - Soft Erosion Hybrid

Same architecture as model 2, with an erosion penalty added to the loss to suppress fragments.

In [ ]:
torch.manual_seed(SEED)
model5 = HybridUNet().to(device)

criterion5 = SoftErosionLoss(dice_weight=0.4, bce_weight=0.4,
                             erosion_weight=0.2, kernel_size=EROSION_KERNEL)
optimizer5 = optim.Adam(model5.parameters(), lr=0.001)
scheduler5 = optim.lr_scheduler.ReduceLROnPlateau(optimizer5, mode='min',
                                                  patience=5, factor=0.5)

print(f"Erosion kernel: {EROSION_KERNEL}")

history5 = train_model(
    model5, hybrid_train_loader, hybrid_val_loader, criterion5,
    optimizer5, scheduler5, MAX_EPOCHS, PATIENCE,
    os.path.join(CKPT_DIR, 'hybrid_soft_erosion.pth'), 'Soft Erosion Hybrid', device,
    PP_KERNEL, EROSION_KERNEL)

## 11. Summary

Best validation loss and Dice for each model, read back from the saved checkpoints.

In [ ]:
checkpoints = [
    ('plain_unet.pth', 'Plain U-Net'),
    ('hybrid_unet.pth', 'Hybrid U-Net'),
    ('octf_backbone.pth', 'OCT-F Backbone'),
    ('modified_octf_2ch.pth', 'Modified OCT-F 2ch'),
    ('hybrid_soft_erosion.pth', 'Soft Erosion Hybrid'),
]

print(f"{'Model':<24} {'Params':<12} {'In':<4} {'Best ep':<9} {'Stopped':<9} "
      f"{'Val loss':<10} {'Val dice':<10} {'Time'}")
print("=" * 100)

for filename, name in checkpoints:
    path = os.path.join(CKPT_DIR, filename)
    if not os.path.exists(path):
        print(f"{name:<24} MISSING")
        continue
    ck = torch.load(path, map_location='cpu', weights_only=False)
    cfg = ck['config']
    total_time = sum(ck['history']['epoch_time'])
    print(f"{name:<24} {cfg['params']:<12,} {cfg['input_channels']:<4} "
          f"{ck['epoch']:<9} {cfg['early_stopped_at']:<9} "
          f"{ck['val_loss']:<10.4f} {ck['val_dice']:<10.4f} {total_time / 60:.1f}m")

print(f"\nCheckpoints saved to {CKPT_DIR}")
print("Next: download checkpoints and run the analysis notebooks locally.")